In [1]:
#calculating and preparing two new columns.
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler

ck_registered = "curricular_units_2nd_sem_enrolled"
ck_passed = "curricular_units_2nd_sem_approved"
ck_source_columns = [
    "curricular_units_2nd_sem_credited", ck_registered, "curricular_units_2nd_sem_evaluations",
    ck_passed, "curricular_units_2nd_sem_grade", "curricular_units_2nd_sem_without_evaluations",
    "unemployment_rate", "inflation_rate", "gdp",
]

def ck_make_features(counts):
    registered = counts[ck_registered]
    return pd.DataFrame({
        "approval_rate_s2": counts[ck_passed] / registered.where(registered > 0),
        "no_registration_s2": registered.eq(0).astype(int),
    }, index=counts.index)

def ck_feature_names(transformer, input_features):
    return ["approval_rate_s2", "no_registration_s2"]

ck_rate_steps = Pipeline([
    ("fill_undefined", SimpleImputer(strategy="median")),
    ("scale", MinMaxScaler()),
])
ck_feature_steps = Pipeline([
    ("calculate", FunctionTransformer(ck_make_features, feature_names_out=ck_feature_names)),
    ("prepare", ColumnTransformer([
        ("rate", ck_rate_steps, ["approval_rate_s2"]),
        ("flag", "passthrough", ["no_registration_s2"]),
    ], verbose_feature_names_out=False)),
])
ck_addon = ("ck_semester2", ck_feature_steps, [ck_registered, ck_passed])

In [2]:
# Show exactly how blocks fit beside existing numerical columns and handles zero registrations.
demo = pd.DataFrame({ck_registered: [6, 6, 6, 0], ck_passed: [6, 3, 0, 0]})
print("Before model preparation:")
print(ck_make_features(demo).to_string(index=False))

demo_preprocessor = ColumnTransformer([
    ("numerical", MinMaxScaler(), [ck_registered, ck_passed]),
    ck_addon,
])
prepared_demo = demo_preprocessor.fit_transform(demo)
print("\nModel-ready columns:")
print(pd.DataFrame(prepared_demo, columns=demo_preprocessor.get_feature_names_out()).to_string(index=False))

Before model preparation:
 approval_rate_s2  no_registration_s2
              1.0                   0
              0.5                   0
              0.0                   0
              NaN                   1

Model-ready columns:
 numerical__curricular_units_2nd_sem_enrolled  numerical__curricular_units_2nd_sem_approved  ck_semester2__approval_rate_s2  ck_semester2__no_registration_s2
                                          1.0                                           1.0                             1.0                               0.0
                                          1.0                                           0.5                             0.5                               0.0
                                          1.0                                           0.0                             0.0                               0.0
                                          0.0                                           0.0                             0.5       

In [3]:
# Verify CK nine original inputs and two additions on the project's seed-42 training rows.
from pathlib import Path
from sklearn.model_selection import train_test_split

folders = [Path("."), Path("../data/clean"), Path("data/clean"),
           Path("../data/processed"), Path("data/processed"), Path("../data/raw"), Path("data/raw")]
names = ["clean_data.csv", "students_cleaned.csv", "data.csv", "data_raw.csv"]
source = next((folder / name for folder in folders for name in names if (folder / name).is_file()), None)

if source is None:
    print("The worked example ran. To run the real-data check too, place the project CSV beside this notebook.")
else:
    students = pd.read_csv(source, sep=None, engine="python", encoding="utf-8-sig")
    students.columns = (students.columns.str.strip().str.lower().str.replace(r"[ /]+", "_", regex=True)
                        .str.replace(r"['()]", "", regex=True))
    status = students["target"].str.strip()
    assert status.isin(["Graduate", "Dropout", "Enrolled"]).all()
    known = status.isin(["Graduate", "Dropout"])
    y = status[known].map({"Graduate": 0, "Dropout": 1})
    train_ids, _ = train_test_split(y.index, test_size=0.2, stratify=y, random_state=42)
    training = students.loc[train_ids, ck_source_columns]

    ck_preprocessor = ColumnTransformer([
        ("numerical", MinMaxScaler(), ck_source_columns),
        ck_addon,
    ])
    prepared = ck_preprocessor.fit_transform(training)
    before = ck_make_features(training)
    assert prepared.shape == (len(training), len(ck_source_columns) + 2)
    assert not pd.isna(prepared).any()
    assert (prepared[:, -1] == before["no_registration_s2"]).all()
    print("Training students:", len(training))
    print("Students with undefined rates:", before["approval_rate_s2"].isna().sum())
    print("Students with measured 0% rates:", before["approval_rate_s2"].eq(0).sum())
    print("Median measured rate in training:", before["approval_rate_s2"].median())
    print("Prepared columns:", prepared.shape[1], "| Blanks:", pd.isna(prepared).sum())

Training students: 2904
Students with undefined rates: 118
Students with measured 0% rates: 514
Median measured rate in training: 0.875
Prepared columns: 11 | Blanks: 0
